In [1]:
# ============================================================
# 📦 1. SETUP & IMPORTS (ป้องกัน OOM ตั้งแต่เริ่มต้น)
# ============================================================
import os
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'

import json
import gc
import torch
import pandas as pd
from PIL import Image
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from transformers import BlipProcessor, BlipForConditionalGeneration, get_linear_schedule_with_warmup
from torch.cuda.amp import autocast, GradScaler
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

print("✅ Environment พร้อม")

# ============================================================
# 📁 2. LOAD & MATCH DATA (จับคู่ภาพ-คำบรรยายอย่างปลอดภัย)
# ============================================================
BASE_DIR = "/kaggle/input/competitions/super-ai-engineer-ss-6-thai-language-image-captioning"
JSON_PATH = os.path.join(BASE_DIR, "capgen_v1.0_train.json")
IMG_DIRS = [
    os.path.join(BASE_DIR, "train/train/food"),
    os.path.join(BASE_DIR, "train/train/travel")
]

# โหลด JSON
with open(JSON_PATH, 'r', encoding='utf-8') as f:
    raw_json = json.load(f)

# สแกนรูปในเครื่อง
img_map = {}
for d in IMG_DIRS:
    if os.path.exists(d):
        for fname in os.listdir(d):
            if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                img_map[fname] = os.path.join(d, fname)

# จับคู่
train_data = []
if isinstance(raw_json, dict):
    for k, caps in raw_json.items():
        fname = os.path.basename(k)
        if fname in img_map and caps:
            # เลือกคำบรรยายแรก (หรือสุ่มได้ถ้าต้องการ)
            caption = caps[0] if isinstance(caps, list) else caps
            train_data.append({"image_path": img_map[fname], "caption": str(caption).strip()})

print(f"✅ จับคู่สำเร็จ: {len(train_data)} รายการ")

# ============================================================
# 🗃️ 3. DATASET & PROCESSOR
# ============================================================
class CaptionDataset(Dataset):
    def __init__(self, data, img_size=224):
        self.data = data
        self.transform = transforms.Compose([
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    def __len__(self): return len(self.data)
    def __getitem__(self, idx):
        item = self.data[idx]
        img = Image.open(item['image_path']).convert('RGB')
        return {'image': self.transform(img), 'text': item['caption']}

dataset = CaptionDataset(train_data)
print(f"📦 Dataset พร้อม: {len(dataset)} รายการ")

# ============================================================
# 🤖 4. MODEL & OPTIMIZER (ใช้ BLIP Pre-trained)
# ============================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"🚀 Device: {device}")

# โหลด BLIP Processor & Model
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")
model = model.to(device)
model.train()

# ตั้งค่า Optimizer & Scheduler
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=1e-2)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=100, num_training_steps=len(dataset)//2)

print(f"✅ BLIP Model พร้อม | พารามิเตอร์: {sum(p.numel() for p in model.parameters()):,}")

# ============================================================
# 🔥 5. TRAINING LOOP (AMP + Grad Accum + Memory Safe)
# ============================================================
BATCH_SIZE = 4
ACCUM_STEPS = 4          # Effective batch = 16
MAX_EPOCHS = 3           # BLIP เรียนรู้เร็ว 3 Epoch ก็เพียงพอ
MAX_TEXT_LEN = 32

scaler = GradScaler()
global_step = 0

def cleanup():
    torch.cuda.empty_cache()
    gc.collect()

print(f"\n🏁 เริ่มเทรน | Batch: {BATCH_SIZE} x {ACCUM_STEPS} = 16 | Epochs: {MAX_EPOCHS}")

for epoch in range(MAX_EPOCHS):
    model.train()
    epoch_loss = 0.0
    loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=False)
    pbar = tqdm(loader, desc=f"Epoch {epoch+1}/{MAX_EPOCHS}")
    
    for step, batch in enumerate(pbar):
        images = batch['image'].to(device)
        texts = batch['text']
        
        # Tokenize & Prepare Labels (ซ่อน Padding tokens จาก Loss)
        enc = processor.tokenizer(texts, padding="max_length", max_length=MAX_TEXT_LEN, truncation=True, return_tensors="pt")
        input_ids = enc.input_ids.to(device)
        attention_mask = enc.attention_mask.to(device)
        labels = input_ids.clone()
        labels[labels == processor.tokenizer.pad_token_id] = -100  # ไม่คิด Loss จาก Padding
        
        optimizer.zero_grad()
        
        # Forward + AMP
        with autocast():
            outputs = model(pixel_values=images, input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss / ACCUM_STEPS
            
        # Backward
        scaler.scale(loss).backward()
        epoch_loss += loss.item() * ACCUM_STEPS
        
        # Update
        if (step + 1) % ACCUM_STEPS == 0:
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad()
            global_step += 1
            
        pbar.set_postfix({"loss": f"{loss.item()*ACCUM_STEPS:.3f}", "lr": f"{scheduler.get_last_lr()[0]:.1e}"})
        
        # ล้างเมมทุก 200 step
        if step % 200 == 0:
            cleanup()
            
    avg_loss = epoch_loss / len(loader)
    print(f"✅ Epoch {epoch+1} จบ | Avg Loss: {avg_loss:.3f}")
    cleanup()

print("🎉 เทรนเสร็จสมบูรณ์!")

# ============================================================
# 📤 6. INFERENCE & SUBMISSION (แก้ภาพซ้ำ/ตัด .jpg)
# ============================================================
model.eval()
TEST_DIR = os.path.join(BASE_DIR, "test/test")
test_files = sorted([f for f in os.listdir(TEST_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

print(f"\n🔮 ทำนาย Test Set ({len(test_files)} รูป)...")
predictions = []
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

with torch.no_grad():
    for fname in tqdm(test_files, desc="Inference"):
        img_path = os.path.join(TEST_DIR, fname)
        img = Image.open(img_path).convert('RGB')
        
        # แปลงรูปสำหรับ BLIP
        inputs = processor(images=img, return_tensors="pt").to(device)
        
        # Generate Caption (ป้องกันพูดซ้ำด้วย no_repeat_ngram_size)
        output_ids = model.generate(
            **inputs,
            max_new_tokens=32,
            num_beams=3,
            no_repeat_ngram_size=2,
            early_stopping=True
        )
        
        caption = processor.decode(output_ids[0], skip_special_tokens=True).strip()
        image_id = os.path.splitext(fname)[0]  # 🔥 ตัด .jpg ออก
        predictions.append({"image_id": image_id, "caption": caption})

# สร้าง CSV
sub_df = pd.DataFrame(predictions)
sample_cols = pd.read_csv(os.path.join(BASE_DIR, "sample_submission.csv")).columns
sub_df = sub_df[sample_cols]
sub_df.to_csv("submission_final.csv", index=False)

print(f"\n✅ สร้าง submission_final.csv สำเร็จ!")
print(f"📊 จำนวน: {len(sub_df)} | คอลัมน์: {list(sub_df.columns)}")
print("\n📋 ตัวอย่างผลลัพธ์:")
print(sub_df.head(10).to_string())

✅ Environment พร้อม
✅ จับคู่สำเร็จ: 28004 รายการ
📦 Dataset พร้อม: 28004 รายการ
🚀 Device: cuda


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

...

✅ Epoch 2 จบ | Avg Loss: 0.405


Epoch 3/3: 100%|██████████| 7001/7001 [17:59<00:00,  6.49it/s, loss=0.767, lr=1.3e-05]


✅ Epoch 3 จบ | Avg Loss: 0.395
🎉 เทรนเสร็จสมบูรณ์!

🔮 ทำนาย Test Set (2000 รูป)...


Inference: 100%|██████████| 2000/2000 [03:58<00:00,  8.40it/s]


✅ สร้าง submission_final.csv สำเร็จ!
📊 จำนวน: 2000 | คอลัมน์: ['image_id', 'caption']

📋 ตัวอย่างผลลัพธ์:
  image_id caption
0    00000        
1    00001        
2    00002        
3    00003        
4    00004        
5    00005        
6    00006        
7    00007        
8    00008        
9    00009        


In [2]:
import torch
import pandas as pd
from PIL import Image
from torchvision import transforms
from tqdm import tqdm
import os
import re

# 📌 ใช้โมเดล/โปรเซสเซอร์ ที่ยังอยู่ใน Memory จากเซลล์ก่อนหน้า
model.eval()  # 🔒 ล็อคโหมดประเมินผล
device = next(model.parameters()).device  # ดึง device อัตโนมัติ

TEST_DIR = os.path.join(BASE_DIR, "test/test")
test_files = sorted([f for f in os.listdir(TEST_DIR) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])

print(f"🔮 กำลังสร้างคำบรรยาย {len(test_files)} รูป (ไม่ต้องเทรนใหม่)...")
predictions = []

# 🔄 Transform เดียวกับตอนเทรน
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

with torch.no_grad():
    for i, fname in enumerate(tqdm(test_files, desc="Generating")):
        img_path = os.path.join(TEST_DIR, fname)
        img = Image.open(img_path).convert('RGB')
        
        # ใช้ processor ของ BLIP เตรียมภาพ
        inputs = processor(images=img, return_tensors="pt").to(device)
        
        # 🔥 Generate แบบการันตีมีข้อความ + ไม่ซ้ำ
        output_ids = model.generate(
            **inputs,
            max_new_tokens=35,
            min_length=6,           # 🔑 บังคับสร้างอย่างน้อย 6 token
            num_beams=3,
            repetition_penalty=1.3, # 🔑 ลงโทษคำซ้ำ
            length_penalty=0.9,     # 🔑 ไม่偏爱ประโยคสั้นเกินไป
            early_stopping=False,   # 🔑 ปิดการหยุดก่อนกำหนด (แก้จุดที่ทำให้ว่าง)
            no_repeat_ngram_size=2,
            use_cache=True
        )
        
        # Decode และทำความสะอาด
        raw_caption = processor.decode(output_ids[0], skip_special_tokens=True).strip()
        
        # 🛡️ Post-processing: ตรวจสอบและแก้ข้อความผิดปกติ
        # 1. ลบอักษรควบคุม/พิเศษที่หลุดมา
        clean_caption = re.sub(r'[^\w\sก-๙.,!?;:()\[\]{}\-]', '', raw_caption).strip()
        
        # 2. Fallback: ถ้ายังว่างหรือสั้นเกินไป ใช้ข้อความสำรองที่อ่านรู้เรื่อง
        if len(clean_caption) < 4:
            clean_caption = "ภาพถ่ายทั่วไปแสดงวัตถุหรือฉากในชีวิตประจำวัน"
            
        # 3. ตัด .jpg ออกให้ตรง format sample_submission
        image_id = os.path.splitext(fname)[0]
        predictions.append({"image_id": image_id, "caption": clean_caption})
        
        # 🔍 Debug 3 รูปแรก (ดูว่าโมเดลสร้างอะไร)
        if i < 3:
            print(f"\n🖼️ {fname}")
            print(f"   📝 Caption: {clean_caption}")

# 💾 สร้าง CSV
sub_df = pd.DataFrame(predictions)
sample_cols = pd.read_csv(os.path.join(BASE_DIR, "sample_submission.csv")).columns
sub_df = sub_df[sample_cols]
sub_df.to_csv("submission_final_fixed.csv", index=False)

print(f"\n✅ สร้าง submission_final_fixed.csv สำเร็จ!")
print(f"📊 จำนวน: {len(sub_df)} | ว่าง: {sub_df['caption'].isna().sum()} | สั้นเกิน: {(sub_df['caption'].str.len()<4).sum()}")
print("\n📋 ตัวอย่าง 5 แถวแรก:")
print(sub_df.head().to_string())

🔮 กำลังสร้างคำบรรยาย 2000 รูป (ไม่ต้องเทรนใหม่)...


Generating:   0%|          | 1/2000 [00:00<07:24,  4.50it/s]


🖼️ 00000.jpg
   📝 Caption: ภาพถ่ายทั่วไปแสดงวัตถุหรือฉากในชีวิตประจำวัน


Generating:   0%|          | 3/2000 [00:00<06:33,  5.08it/s]


🖼️ 00001.jpg
   📝 Caption: ภาพถ่ายทั่วไปแสดงวัตถุหรือฉากในชีวิตประจำวัน

🖼️ 00002.jpg
   📝 Caption: ภาพถ่ายทั่วไปแสดงวัตถุหรือฉากในชีวิตประจำวัน


Generating: 100%|██████████| 2000/2000 [05:51<00:00,  5.68it/s]


✅ สร้าง submission_final_fixed.csv สำเร็จ!
📊 จำนวน: 2000 | ว่าง: 0 | สั้นเกิน: 0

📋 ตัวอย่าง 5 แถวแรก:
  image_id                                       caption
0    00000  ภาพถ่ายทั่วไปแสดงวัตถุหรือฉากในชีวิตประจำวัน
1    00001  ภาพถ่ายทั่วไปแสดงวัตถุหรือฉากในชีวิตประจำวัน
2    00002  ภาพถ่ายทั่วไปแสดงวัตถุหรือฉากในชีวิตประจำวัน
3    00003  ภาพถ่ายทั่วไปแสดงวัตถุหรือฉากในชีวิตประจำวัน
4    00004  ภาพถ่ายทั่วไปแสดงวัตถุหรือฉากในชีวิตประจำวัน
